# RAG ingestion

Jednorazowy pipeline do dodawania nowego dokumentu miejskiego do bazy Pismaka.

In [ ]:
from pathlib import Path
import os, sys, json

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "docker-compose.yaml").exists()
        and (candidate / "src").is_dir()
    ),
    Path.cwd(),
)
SRC = PROJECT_ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

from Zrodlowyszukiwator.rag_ingestion.catalog import load_catalog
from Zrodlowyszukiwator.rag_ingestion.pipeline import ingest_pdf
from Zrodlowyszukiwator.rag_ingestion.embedder import OllamaEmbedder
from Zrodlowyszukiwator.rag_ingestion.repository import PgVectorRepository

print("Gotowe.")

In [ ]:
# Konfiguracja.
# Wskaż jeden plik dodawany w tej sesji.

POLICIES_DIR = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "RAW"
CATALOG_PATH = PROJECT_ROOT / "src" / "Zrodlowyszukiwator" / "RAG" / "document_catalog.json"
POLICIES_LIST = list(POLICIES_DIR.iterdir())
POLICIES_LIST

In [ ]:
catalog = load_catalog(CATALOG_PATH)

In [ ]:
DATABASE_URL = os.environ["RAG_INGESTION_DATABASE_URL"]
EMBEDDING_DIM = int(os.getenv("RAG_EMBEDDING_DIM", "1024"))

repo = PgVectorRepository(
    DATABASE_URL,
    embedding_dim=EMBEDDING_DIM,
)
repo.ensure_schema()

print("Schema gotowy.")

In [ ]:
embedder = OllamaEmbedder(
    model=os.getenv("RAG_EMBEDDING_MODEL", "bge-m3"),
    base_url=os.getenv("OLLAMA_URL", "http://127.0.0.1:11434"),
    batch_size=16,
)

test_embedding = embedder.embed(["Test embeddingu dla dokumentów miejskich."])[0]
print("Wymiar embeddingu:", len(test_embedding))

if len(test_embedding) != EMBEDDING_DIM:
    raise ValueError(
        f"Model zwraca {len(test_embedding)} wymiarów, "
        f"a DB jest skonfigurowana na {EMBEDDING_DIM}."
    )

In [ ]:
# wrzuć wszystkie do bazy danych.

for file in POLICIES_LIST:
    if file.suffix.lower() == ".pdf":
        print(f"Znaleziono plik PDF: {file.name}")


        if file.name not in catalog:
            raise KeyError(
                f"Brak metadanych dla {file.name!r} w {CATALOG_PATH}"
            )

        metadata = catalog[file.name]
        display(metadata)

        result = ingest_pdf(
            file,
            metadata,
            repository=repo,
            embedder=embedder,
            max_chars=3500,
            min_chars=250,
            overlap_units=1,
            force=False,
        )

result

In [ ]:
# Kontrola bazy.

doc_count, chunk_count = repo.counts()

print("Dokumenty w bazie:", doc_count)
print("Chunki w bazie:", chunk_count)